# 01 - Cleaning and Integration
This notebook covers Deliverable A: Data Cleaning & Integration Pipeline.


In [ ]:
import os, sys
import pandas as pd
import numpy as np

# DYNAMIC PATH RESOLUTION
def find_project_root():
    curr = os.path.abspath(os.getcwd())
    while curr != os.path.dirname(curr):
        if os.path.exists(os.path.join(curr, 'data', 'raw', 'crop_yield_train.csv')):
            return curr
        if os.path.exists(os.path.join(curr, 'team_6', 'data', 'raw', 'crop_yield_train.csv')):
            return os.path.join(curr, 'team_6')
        curr = os.path.dirname(curr)
    return os.path.abspath(os.getcwd())

ROOT_DIR = find_project_root()
RAW_DIR = os.path.join(ROOT_DIR, 'data', 'raw')
PROCESSED_DIR = os.path.join(ROOT_DIR, 'data', 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

required = ['crop_yield_train.csv', 'regional_weather.csv', 'market_prices.csv']
for f in required:
    p = os.path.join(RAW_DIR, f)
    if not os.path.exists(p):
        print(f"FAIL: Missing {p}")
        sys.exit(1)
print(f"PASS: All raw data files located successfully in '{RAW_DIR}'!")


## A1 & A2: Load and Clean Data


In [ ]:
# Load datasets
train = pd.read_csv(os.path.join(RAW_DIR, 'crop_yield_train.csv'))
test = pd.read_csv(os.path.join(RAW_DIR, 'crop_yield_leaderboard_test.csv'))
weather = pd.read_csv(os.path.join(RAW_DIR, 'regional_weather.csv'))
prices = pd.read_csv(os.path.join(RAW_DIR, 'market_prices.csv'))

# Cleaning functions
def clean_plots(df):
    df = df.copy()
    df.replace([-999, '-999', -999.0], np.nan, inplace=True)
    if 'region' in df.columns:
        df['region'] = df['region'].astype(str).str.lower().str.strip()
    if 'crop_type' in df.columns:
        df['crop_type'] = df['crop_type'].astype(str).str.lower().str.strip()
    return df

def clean_weather(df):
    df = df.copy()
    df.replace([-999, '-999', -999.0], np.nan, inplace=True)
    df['region'] = df['region'].astype(str).str.lower().str.strip()
    month_map = {'jan':1, 'feb':2, 'mar':3, 'apr':4, 'may':5, 'jun':6, 'jul':7, 'aug':8, 'sep':9, 'oct':10, 'nov':11, 'dec':12}
    if 'month' in df.columns:
        df['month'] = df['month'].astype(str).str.lower().str.slice(0,3).map(month_map)
        df['month'] = pd.to_numeric(df['month'], errors='coerce')
    return df

def clean_prices(df):
    df = df.copy()
    df.replace([-999, '-999', -999.0], np.nan, inplace=True)
    df['region'] = df['region'].astype(str).str.lower().str.strip()
    df['crop_type'] = df['crop_type'].astype(str).str.lower().str.strip()
    if 'price_birr_per_quintal' in df.columns:
        df.loc[df['price_birr_per_quintal'] > 20000, 'price_birr_per_quintal'] /= 10
    return df

train_clean = clean_plots(train)
test_clean = clean_plots(test)
weather_clean = clean_weather(weather)
prices_clean = clean_prices(prices)

print("Cleaned train shape:", train_clean.shape)
print("Cleaned test shape:", test_clean.shape)


## A3 & A4: Join Weather & Market Prices


In [ ]:
# Join Weather & Prices to master datasets
def join_tables(plots, weather, prices):
    df = plots.copy()
    # Weather seasonal aggregation (4 months from planting_month)
    weather_agg_list = []
    for idx, row in df.iterrows():
        reg = row['region']
        yr = row['survey_year']
        pm = row['planting_month']
        if pd.isna(pm):
            months = [5, 6, 7, 8]
        else:
            pm = int(pm)
            months = [(pm + i - 1) % 12 + 1 for i in range(4)]
        sub_w = weather[(weather['region'] == reg) & (weather['year'] == yr) & (weather['month'].isin(months))]
        if len(sub_w) > 0:
            t_mean = sub_w['avg_temp_c'].mean()
            r_sum = sub_w['monthly_rainfall_mm'].sum()
            h_sum = sub_w['extreme_heat_days'].sum()
        else:
            t_mean, r_sum, h_sum = np.nan, np.nan, np.nan
        weather_agg_list.append({
            'season_temp_mean': t_mean,
            'season_rainfall_sum': r_sum,
            'season_extreme_heat_days': h_sum
        })
    w_df = pd.DataFrame(weather_agg_list)
    df = pd.concat([df, w_df], axis=1)
    
    # Merge market prices
    df = pd.merge(df, prices[['region', 'crop_type', 'year', 'price_birr_per_quintal']], 
                  left_on=['region', 'crop_type', 'survey_year'], 
                  right_on=['region', 'crop_type', 'year'], 
                  how='left')
    if 'year' in df.columns:
        df.drop(columns=['year'], inplace=True)
    return df

master_train = join_tables(train_clean, weather_clean, prices_clean)
master_test = join_tables(test_clean, weather_clean, prices_clean)

master_train.to_csv(os.path.join(PROCESSED_DIR, 'master_train.csv'), index=False)
master_test.to_csv(os.path.join(PROCESSED_DIR, 'master_test.csv'), index=False)
print("SUCCESS: master_train.csv and master_test.csv saved!")
